# Práctica 7. Agrupamiento de datos con K-Means

## Objetivo

Crear y evaluar un modelo de **K-Means** para agrupar recetas según el comportamiento agregado de sus reseñas. A diferencia de la clasificación, no se proporciona una etiqueta objetivo: el algoritmo busca grupos internamente similares.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RUTA_DATOS = "dataset_limpio.csv"
df = pd.read_csv(RUTA_DATOS)
df["created_at"] = pd.to_datetime(df["created_at"])
df.info()
df.head()


## 1. Agregación por receta

La unidad de análisis será la receta, no cada comentario. Se calculan cantidad de reseñas, promedios y totales de interacción, manteniendo el enfoque descriptivo de las prácticas anteriores.

In [ ]:
recetas = (
    df.groupby(["recipe_code", "recipe_name"], as_index=False)
      .agg(
          numero_resenas=("comment_id", "count"),
          promedio_estrellas=("stars", lambda s: s[s > 0].mean()),
          promedio_best_score=("best_score", "mean"),
          total_me_gusta=("thumbs_up", "sum"),
          total_no_me_gusta=("thumbs_down", "sum"),
          total_respuestas=("reply_count", "sum"),
          promedio_reputacion=("user_reputation", "mean")
      )
)
recetas = recetas.fillna(0)
recetas.shape, recetas.head()


In [ ]:
caracteristicas = [
    "numero_resenas", "promedio_estrellas", "promedio_best_score",
    "total_me_gusta", "total_no_me_gusta", "total_respuestas",
    "promedio_reputacion"
]
X_cluster = recetas[caracteristicas].copy()

# Las variables de conteo tienen escalas muy distintas; se transforman antes de estandarizar.
X_transformado = np.log1p(X_cluster)
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X_escalado = StandardScaler().fit_transform(X_transformado)


## 2. Elección de k

Se emplean dos criterios complementarios: la **inercia** (método del codo) y el **coeficiente de silueta**. La decisión final debe considerar ambos y también si los grupos son interpretables.

In [ ]:
inercias = []
siluetas = []
valores_k = range(2, 9)
for k in valores_k:
    modelo = KMeans(n_clusters=k, random_state=42, n_init=20)
    etiquetas = modelo.fit_predict(X_escalado)
    inercias.append(modelo.inertia_)
    siluetas.append(silhouette_score(X_escalado, etiquetas))

resultados_k = pd.DataFrame({"k": list(valores_k), "inercia": inercias, "silueta": siluetas})
resultados_k


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(resultados_k["k"], resultados_k["inercia"], marker="o")
ax[0].set_title("Método del codo")
ax[0].set_xlabel("k")
ax[0].set_ylabel("Inercia")
ax[1].plot(resultados_k["k"], resultados_k["silueta"], marker="o", color="darkorange")
ax[1].set_title("Coeficiente de silueta")
ax[1].set_xlabel("k")
ax[1].set_ylabel("Silueta promedio")
plt.tight_layout()
plt.show()


In [ ]:
k_elegido = int(resultados_k.loc[resultados_k["silueta"].idxmax(), "k"])
modelo_kmeans = KMeans(n_clusters=k_elegido, random_state=42, n_init=20)
recetas["cluster"] = modelo_kmeans.fit_predict(X_escalado)
print(f"k elegido por máxima silueta: {k_elegido}")
print(f"Silueta final: {silhouette_score(X_escalado, recetas['cluster']):.4f}")
print("\nTamaño de los clusters:")
print(recetas["cluster"].value_counts().sort_index())


In [ ]:
perfil_clusters = recetas.groupby("cluster")[caracteristicas].mean().round(2)
perfil_clusters


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
coordenadas = pca.fit_transform(X_escalado)
plt.figure(figsize=(8, 5))
for cluster_id in sorted(recetas["cluster"].unique()):
    seleccion = recetas["cluster"] == cluster_id
    plt.scatter(coordenadas[seleccion, 0], coordenadas[seleccion, 1], s=70, label=f"Cluster {cluster_id}")
plt.title("Visualización de clusters mediante PCA")
plt.xlabel("Componente principal 1")
plt.ylabel("Componente principal 2")
plt.legend(title="Cluster")
plt.tight_layout()
plt.show()


## 3. Conclusiones

- K-Means agrupó recetas con patrones semejantes de volumen, calificación e interacción.
- La transformación `log1p` reduce el efecto de valores extremos en los totales.
- La silueta ayuda a evaluar separación, pero el perfil promedio de cada grupo es indispensable para interpretarlo.
- Los clusters no son categorías verdaderas: dependen de las variables elegidas, la escala y el valor de `k`.